# Credit Card Default Prediction
Predict next-month credit-card default (`0 = No Default`, `1 = Default`).

This notebook includes EDA, preprocessing, baseline models, class-imbalance handling with `class_weight="balanced"`, confusion matrices, model comparison, and Random Forest feature importance.

## 1. Libraries

In [ ]:
!pip install xlrd -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, ConfusionMatrixDisplay
)

## 2. Load dataset

In [ ]:
df = pd.read_excel("default of credit card clients.xls", header=1)
df.head()

## 3. Inspect dataset

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())

In [ ]:
df.info()

## 4. Target and class imbalance

In [ ]:
print(df["default payment next month"].value_counts())
print("\nPercentage:")
print(df["default payment next month"].value_counts(normalize=True).mul(100))

In [ ]:
df["default payment next month"].value_counts().plot(kind="bar", figsize=(6,4))
plt.title("Default Payment Distribution")
plt.xlabel("Default Payment Next Month")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.show()

## 5. Numerical EDA

In [ ]:
num_cols = [
    "LIMIT_BAL","AGE",
    "BILL_AMT1","BILL_AMT2","BILL_AMT3","BILL_AMT4","BILL_AMT5","BILL_AMT6",
    "PAY_AMT1","PAY_AMT2","PAY_AMT3","PAY_AMT4","PAY_AMT5","PAY_AMT6"
]
df[num_cols].describe()

In [ ]:
df.groupby("default payment next month")[num_cols].mean()

## 6. Categorical and payment-history EDA

In [ ]:
for col in ["SEX","EDUCATION","MARRIAGE","PAY_0"]:
    print(f"--- {col} ---")
    print(df[col].value_counts().sort_index())
    print()

In [ ]:
for col in ["SEX","EDUCATION","MARRIAGE","PAY_0"]:
    print(f"Default rate by {col}:")
    print(df.groupby(col)["default payment next month"].mean())
    print()

## 7. Define X and y

In [ ]:
df = df.drop(columns=["ID"])
X = df.drop(columns=["default payment next month"])
y = df["default payment next month"]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 8. Feature types

In [ ]:
categorical_cols = [
    "SEX","EDUCATION","MARRIAGE",
    "PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6"
]

numerical_cols = [
    "LIMIT_BAL","AGE",
    "BILL_AMT1","BILL_AMT2","BILL_AMT3","BILL_AMT4","BILL_AMT5","BILL_AMT6",
    "PAY_AMT1","PAY_AMT2","PAY_AMT3","PAY_AMT4","PAY_AMT5","PAY_AMT6"
]

## 9. Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training:", X_train.shape)
print("Testing:", X_test.shape)

## 10. Preprocessing

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ]
)

# PART A — BASELINE MODELS

## 11. Baseline models

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

decision_tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    ))
])

logistic_model.fit(X_train, y_train)
decision_tree_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)

## 12. Evaluation function

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    print("=" * 60)
    print(model_name)
    print("=" * 60)
    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    }

## 13. Baseline results

In [ ]:
baseline_results = [
    evaluate_model(logistic_model, X_test, y_test, "Logistic Regression"),
    evaluate_model(decision_tree_model, X_test, y_test, "Decision Tree"),
    evaluate_model(random_forest_model, X_test, y_test, "Random Forest")
]
baseline_results_df = pd.DataFrame(baseline_results)
baseline_results_df

## 14. Baseline confusion matrices

In [ ]:
baseline_models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": decision_tree_model,
    "Random Forest": random_forest_model
}

fig, axes = plt.subplots(1, 3, figsize=(16,4))
for ax, (name, model) in zip(axes, baseline_models.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, model.predict(X_test), ax=ax)
    ax.set_title(name)
plt.tight_layout()
plt.show()

# PART B — CLASS IMBALANCE HANDLING

We use `class_weight="balanced"`. This gives more importance to the minority default class during training without creating synthetic observations.

## 15. Balanced models

In [ ]:
balanced_logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=1000))
])

balanced_decision_tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        class_weight="balanced", random_state=42
    ))
])

balanced_random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200, class_weight="balanced",
        random_state=42, n_jobs=-1
    ))
])

balanced_logistic_model.fit(X_train, y_train)
balanced_decision_tree_model.fit(X_train, y_train)
balanced_random_forest_model.fit(X_train, y_train)

## 16. Balanced results

In [ ]:
balanced_results = [
    evaluate_model(
        balanced_logistic_model, X_test, y_test,
        "Balanced Logistic Regression"
    ),
    evaluate_model(
        balanced_decision_tree_model, X_test, y_test,
        "Balanced Decision Tree"
    ),
    evaluate_model(
        balanced_random_forest_model, X_test, y_test,
        "Balanced Random Forest"
    )
]
balanced_results_df = pd.DataFrame(balanced_results)
balanced_results_df

## 17. Balanced confusion matrices

In [ ]:
balanced_models = {
    "Balanced Logistic Regression": balanced_logistic_model,
    "Balanced Decision Tree": balanced_decision_tree_model,
    "Balanced Random Forest": balanced_random_forest_model
}

fig, axes = plt.subplots(1, 3, figsize=(16,4))
for ax, (name, model) in zip(axes, balanced_models.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, model.predict(X_test), ax=ax)
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 18. Before vs after comparison

In [ ]:
comparison = pd.concat([
    baseline_results_df.assign(Type="Original"),
    balanced_results_df.assign(Type="Balanced")
], ignore_index=True)

comparison[
    ["Model","Type","Accuracy","Precision","Recall","F1","ROC-AUC"]
]

## 19. Comparison chart

In [ ]:
comparison_plot = comparison.set_index(["Model","Type"])[
    ["Accuracy","Precision","Recall","F1","ROC-AUC"]
]
comparison_plot.plot(kind="bar", figsize=(14,6))
plt.title("Original vs Balanced Model Performance")
plt.ylabel("Score")
plt.ylim(0,1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# PART C — RANDOM FOREST FEATURE IMPORTANCE

## 20. Feature importance

In [ ]:
rf_preprocessor = balanced_random_forest_model.named_steps["preprocessor"]
rf_classifier = balanced_random_forest_model.named_steps["classifier"]

feature_names = rf_preprocessor.get_feature_names_out()
importance = rf_classifier.feature_importances_

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
}).sort_values("Importance", ascending=False)

feature_importance.head(15)

## 21. Top 15 feature-importance plot

In [ ]:
top_features = feature_importance.head(15).sort_values("Importance")

plt.figure(figsize=(10,7))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 15 Random Forest Features")
plt.tight_layout()
plt.show()

## 22. Final comparison table

In [ ]:
final_results = comparison[
    ["Model","Type","Accuracy","Precision","Recall","F1","ROC-AUC"]
].sort_values("F1", ascending=False)

final_results

# Final Conclusion

Use the actual `final_results` table to discuss the model results.

Focus on:
- class imbalance in the target
- Recall and F1 for the default class
- ROC-AUC
- confusion matrices, especially false negatives
- how `class_weight="balanced"` changed performance
- important predictors from Random Forest

Do not claim a model is best until the actual results have been examined.